In [1]:
import numpy as np
import cv2
import dlib
import face_recognition

print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("dlib:", dlib.__version__)
print("face_recognition: OK")

NumPy: 2.5.2
OpenCV: 5.0.0
dlib: 19.24.2
face_recognition: OK


In [5]:
import os
import cv2
import numpy as np

for filename in os.listdir("students"):

    if filename.lower().endswith((".jpg", ".jpeg", ".png")):

        path = os.path.join("students", filename)
        image = cv2.imread(path)

        if image is None:
            print(f"❌ Cannot read: {filename}")
        else:
            print(
                f"✅ {filename} | "
                f"shape={image.shape} | "
                f"dtype={image.dtype}"
            )

✅ image1.jpg | shape=(720, 1280, 3) | dtype=uint8
✅ image2.jpg | shape=(720, 1280, 3) | dtype=uint8
✅ image3.jpg | shape=(720, 1280, 3) | dtype=uint8


In [1]:
import cv2
import dlib
import numpy as np

image = cv2.imread("students/image1.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = np.ascontiguousarray(image, dtype=np.uint8)

print("Image:", image.shape)
print("dtype:", image.dtype)
print("NumPy:", np.__version__)
print("dlib:", dlib.__version__)

detector = dlib.get_frontal_face_detector()

faces = detector(image, 1)

print("Faces detected:", len(faces))

Image: (720, 1280, 3)
dtype: uint8
NumPy: 1.26.4
dlib: 19.24.2
Faces detected: 2


In [3]:
import os
import cv2
import numpy as np
import face_recognition


# ==========================================
# 1. LOAD STUDENT FACE ENCODINGS
# ==========================================

known_encodings = []
known_names = []

for filename in os.listdir("students"):

    if filename.lower().endswith((".jpg", ".jpeg", ".png")):

        path = os.path.join("students", filename)

        image = cv2.imread(path)

        if image is None:
            print(f"Could not read: {filename}")
            continue

        # BGR → RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Ensure correct format
        image = np.ascontiguousarray(image, dtype=np.uint8)

        # Detect faces
        locations = face_recognition.face_locations(image)

        if len(locations) == 0:
            print(f"No face found: {filename}")
            continue

        # Generate face encodings
        encodings = face_recognition.face_encodings(
            image,
            known_face_locations=locations
        )

        if len(encodings) > 0:

            known_encodings.append(encodings[0])

            name = os.path.splitext(filename)[0]
            known_names.append(name)

            print(f"Encoded: {name}")

print("\nTotal students:", len(known_names))


# ==========================================
# 2. START CAMERA
# ==========================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Could not open camera.")
    exit()

print("\nCamera started.")
print("Press Q to quit.")


# ==========================================
# 3. FACE RECOGNITION LOOP
# ==========================================

while True:

    ret, frame = cap.read()

    if not ret:
        print("Camera error.")
        break

    # Resize for faster processing
    small_frame = cv2.resize(
        frame,
        (0, 0),
        fx=0.25,
        fy=0.25
    )

    # BGR → RGB
    rgb_frame = cv2.cvtColor(
        small_frame,
        cv2.COLOR_BGR2RGB
    )

    # Ensure correct format
    rgb_frame = np.ascontiguousarray(
        rgb_frame,
        dtype=np.uint8
    )

    # Detect faces
    locations = face_recognition.face_locations(
        rgb_frame
    )

    # Generate encodings
    encodings = face_recognition.face_encodings(
        rgb_frame,
        known_face_locations=locations
    )

    # Process every detected face
    for location, encoding in zip(locations, encodings):

        top, right, bottom, left = location

        # Compare with known students
        matches = face_recognition.compare_faces(
            known_encodings,
            encoding,
            tolerance=0.5
        )

        name = "Unknown"

        if True in matches:

            index = matches.index(True)

            name = known_names[index]

        # Scale coordinates back to original frame
        top *= 4
        right *= 4
        bottom *= 4
        left *= 4

        # Green = recognized
        # Red = unknown
        if name != "Unknown":
            color = (0, 255, 0)
        else:
            color = (0, 0, 255)

        # Draw face rectangle
        cv2.rectangle(
            frame,
            (left, top),
            (right, bottom),
            color,
            2
        )

        # Draw name
        cv2.putText(
            frame,
            name,
            (left, top - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2
        )

    # Display camera
    cv2.imshow(
        "Student Face Recognition",
        frame
    )

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# ==========================================
# 4. CLEANUP
# ==========================================

cap.release()
cv2.destroyAllWindows()

Encoded: Emaan
Encoded: safoora
Encoded: Tayyaba

Total students: 3

Camera started.
Press Q to quit.


KeyboardInterrupt: 